# Topic Analysis: 368 Topics Across Popularity Tiers

**Scope:** Topic-level analysis using BERTopic probabilities for all 368 topics. Statistical comparisons will use all three tiers (**Top / Middle / Trash**); pairwise contrasts (e.g., Top vs Trash) can be added as needed. Category-level statistics (taxonomy groups, Radway phases) are **excluded** here and will be handled in separate notebooks.

**Data sources (absolute paths):**
- Book features (wide): `/home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/correlation_analysis/data_preparation/book_features/book_taxonomy_main_props_wide.parquet`
- Book features (long): `/home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/correlation_analysis/data_preparation/book_features/book_taxonomy_main_props_long.parquet`
- Book topic probs: `/home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/correlation_analysis/data_preparation/topic_probabilities/book_topic_probs.parquet`
- Chapter topic probs: `/home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/correlation_analysis/data_preparation/topic_probabilities/chapter_topic_probs.parquet`
- Topic lookup (REQUIRED): `/home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/correlation_analysis/data_preparation/taxonomy_radway_eda/topic_lookup.parquet` - Required for topic labels (used instead of topic IDs in all statistical outputs)

**Outputs:** `results/correlation_analysis/01_topic_analysis/` (figures, tables)

## Roadmap (Top / Middle / Trash)

1) Setup & paths
2) Load data + integrity checks (prob sums per book/segment, n_topics=368)
3) **Merge labels from topic_lookup** (REQUIRED - labels used instead of topic IDs in all outputs)
4) Rating-tier prep: map rating_class → {top, middle, trash}; use all three tiers; optional pairwise views (Top vs Trash) if needed
5) Helper utilities: plotting helper, Cliff's delta, prevalence metrics
6) Topic health metrics (prevalence, mass, concentration) — using labels, not taxonomy columns
7) Topic-level distributions & summaries (Top vs Middle vs Trash): medians, means, effect sizes, FDR-corrected tests — **all outputs use labels**
8) Leaderboards & exports (tables + optional plot stubs) — **labeled by topic labels**
9) Optional: per-topic visualization hooks (violin/ECDF/box) for selected topic labels

In [1]:
# 1. Setup & imports
# NOTE: Always use venv for Python commands
# If running from terminal: source venv/bin/activate

from __future__ import annotations

import os
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import mannwhitneyu, kruskal
from statsmodels.stats.multitest import multipletests

# Set plotting defaults
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Inline plotting
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import warnings
warnings.filterwarnings('ignore', message='.*IProgress not found.*')
warnings.filterwarnings('ignore', category=UserWarning, module='tqdm.auto')

PROJECT_ROOT = Path("/home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor")
print(f"✓ PROJECT_ROOT: {PROJECT_ROOT}")

✓ PROJECT_ROOT: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor


## Generate topic_lookup.parquet (if missing)

If `topic_lookup.parquet` is missing, run this command in terminal (with venv activated):

```bash
source venv/bin/activate
python src/correlation_analysis/data_preparation/01_data_validation_extraction.py \
    --output-dir results/correlation_analysis/data_preparation/taxonomy_radway_eda \
    --excluded-book-ids notebooks/07_analysis/statistical_analysis/excluded_book_ids.csv
```

This will generate `topic_lookup.parquet` with topic labels from Stage 08 LLM labeling.



In [2]:
# 2. Paths
DATA_PREP_DIR = PROJECT_ROOT / "results" / "correlation_analysis" / "data_preparation"
BOOK_FEATURES_DIR = DATA_PREP_DIR / "book_features"
TOPIC_PROBS_DIR = DATA_PREP_DIR / "topic_probabilities"
TAXONOMY_RADWAY_DIR = DATA_PREP_DIR / "taxonomy_radway_eda"

# Data file paths
BOOK_WIDE_PATH = BOOK_FEATURES_DIR / "book_taxonomy_main_props_wide.parquet"
BOOK_LONG_PATH = BOOK_FEATURES_DIR / "book_taxonomy_main_props_long.parquet"
BOOK_TOPIC_PROBS_PATH = TOPIC_PROBS_DIR / "book_topic_probs.parquet"
CHAPTER_TOPIC_PROBS_PATH = TOPIC_PROBS_DIR / "chapter_topic_probs.parquet"
TOPIC_LOOKUP_PATH = TAXONOMY_RADWAY_DIR / "topic_lookup.parquet"
GOODREADS_PATH = PROJECT_ROOT / "data" / "processed" / "goodreads.csv"

# Output directories
OUTPUT_DIR = PROJECT_ROOT / "results" / "correlation_analysis" / "topic_analysis"
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
for d in [OUTPUT_DIR, FIG_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Data paths:")
print(f"  Book features (wide): {BOOK_WIDE_PATH}")
print(f"  Book features (long): {BOOK_LONG_PATH}")
print(f"  Book topic probs: {BOOK_TOPIC_PROBS_PATH}")
print(f"  Chapter topic probs: {CHAPTER_TOPIC_PROBS_PATH}")
print(f"  Topic lookup: {TOPIC_LOOKUP_PATH}")
print(f"Outputs: {OUTPUT_DIR}")

Data paths:
  Book features (wide): /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/correlation_analysis/data_preparation/book_features/book_taxonomy_main_props_wide.parquet
  Book features (long): /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/correlation_analysis/data_preparation/book_features/book_taxonomy_main_props_long.parquet
  Book topic probs: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/correlation_analysis/data_preparation/topic_probabilities/book_topic_probs.parquet
  Chapter topic probs: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/correlation_analysis/data_preparation/topic_probabilities/chapter_topic_probs.parquet
  Topic lookup: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/correlation_analysis/data_p

In [3]:
# 3c. Extract keywords from BERTopic model and add to topic_lookup
# Keywords are stored in the BERTopic model's topic_representations_ attribute

print("=" * 80)
print("Extracting keywords from BERTopic model:")
print("=" * 80)

# Try to find and load the BERTopic model
try:
    from bertopic import BERTopic

    # Common model paths to try
    model_candidates = [
        PROJECT_ROOT / "models" / "retrained" / "paraphrase-MiniLM-L6-v2" / "stage09_category_mapping" / "model_1_with_radway_mappings",
        PROJECT_ROOT / "models" / "paraphrase-MiniLM-L6-v2" / "stage09_category_mapping" / "model_1_with_radway_mappings",
        PROJECT_ROOT / "models" / "retrained" / "paraphrase-MiniLM-L6-v2" / "bertopic_model_with_llm_labels",
        PROJECT_ROOT / "models" / "paraphrase-MiniLM-L6-v2" / "bertopic_model_with_llm_labels",
    ]

    model = None
    model_path = None

    for candidate in model_candidates:
        if candidate.exists():
            try:
                print(f"  Trying to load model from: {candidate}")
                model = BERTopic.load(str(candidate))
                model_path = candidate
                print(f"  ✓ Model loaded successfully")
                break
            except Exception as e:
                print(f"  ⚠️  Failed to load from {candidate}: {e}")
                continue

    if model is None:
        print("\n⚠️  Could not find or load BERTopic model.")
        print("  Keywords will not be available.")
        print("  To add keywords, either:")
        print("    1. Re-run: python src/correlation_analysis/data_preparation/01_data_validation_extraction.py")
        print("    2. Or ensure the BERTopic model is available at one of the expected paths")
    else:
        # Extract keywords from model
        if hasattr(model, "topic_representations_"):
            print(f"\n  Extracting keywords from model...")
            keywords_dict = {}

            topic_ids = [tid for tid in model.topic_representations_.keys() if tid != -1]
            for topic_id in sorted(topic_ids):
                if topic_id in model.topic_representations_:
                    kws = model.topic_representations_[topic_id]
                    # Format: list of (word, score) tuples
                    keyword_list = [kw[0] for kw in kws[:10]]  # Top 10 keywords
                    keywords_dict[topic_id] = ", ".join(keyword_list)

            # Add keywords to topic_lookup

            # Check if topic_lookup is available
            if 'topic_lookup' not in globals():
                print("\n⚠️  topic_lookup not yet loaded. Skipping keyword extraction.")
                print("  Run the data loading cell first, then re-run this cell if needed.")
            else:
                if "keywords" not in topic_lookup.columns:
                    topic_lookup["keywords"] = None

                topic_lookup["keywords"] = topic_lookup["topic_id"].map(keywords_dict)

                n_with_keywords = topic_lookup["keywords"].notna().sum()
                print(f"  ✓ Extracted keywords for {n_with_keywords} / {len(topic_lookup)} topics")
                print(f"  ✓ Keywords added to topic_lookup")

                # Save updated topic_lookup
                topic_lookup.to_parquet(TOPIC_LOOKUP_PATH, index=False)
                print(f"  ✓ Saved updated topic_lookup to: {TOPIC_LOOKUP_PATH}")
        else:
            print("\n⚠️  Model does not have topic_representations_ attribute")
            print("  Keywords cannot be extracted from this model")

except ImportError:
    print("\n⚠️  BERTopic not available. Cannot extract keywords.")
    print("  Install with: pip install bertopic")
except Exception as e:
    print(f"\n⚠️  Error extracting keywords: {e}")
    print("  Keywords will not be available in saved tables")



Extracting keywords from BERTopic model:
  Trying to load model from: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/models/retrained/paraphrase-MiniLM-L6-v2/stage09_category_mapping/model_1_with_radway_mappings
  ✓ Model loaded successfully

  Extracting keywords from model...

⚠️  topic_lookup not yet loaded. Skipping keyword extraction.
  Run the data loading cell first, then re-run this cell if needed.


In [4]:
# 3. Load data
book_wide = pd.read_parquet(BOOK_WIDE_PATH)
book_long = pd.read_parquet(BOOK_LONG_PATH)
book_topic_probs = pd.read_parquet(BOOK_TOPIC_PROBS_PATH)
chapter_topic_probs = pd.read_parquet(CHAPTER_TOPIC_PROBS_PATH)

# Topic lookup is REQUIRED for labels (used instead of topic IDs in statistical analysis)
if not TOPIC_LOOKUP_PATH.exists():
    raise FileNotFoundError(
        f"topic_lookup.parquet is REQUIRED but not found at {TOPIC_LOOKUP_PATH}\n"
        f"Please run: python src/correlation_analysis/data_preparation/01_data_validation_extraction.py"
    )

topic_lookup = pd.read_parquet(TOPIC_LOOKUP_PATH)
print("✓ Loaded topic_lookup (REQUIRED for labels)")

# Verify topic_lookup has label column
if "label" not in topic_lookup.columns:
    raise ValueError("topic_lookup must have 'label' column. Check that Stage 08 labels were integrated.")

    # Try to load keywords and rationale from full_model_data if available
    full_model_data_path = TAXONOMY_RADWAY_DIR / "full_model_data.parquet"
    if full_model_data_path.exists():
        print("\nLoading keywords and rationale from full_model_data...")
        full_model_data = pd.read_parquet(full_model_data_path)
        
        # Merge keywords and rationale into topic_lookup
        cols_to_merge = ["topic_id"]
        if "keywords" in full_model_data.columns:
            cols_to_merge.append("keywords")
        if "label_rationale" in full_model_data.columns:
            cols_to_merge.append("label_rationale")
        if "scene_summary" in full_model_data.columns and "scene_summary" not in topic_lookup.columns:
            cols_to_merge.append("scene_summary")
        if "label_is_noise" in full_model_data.columns and "label_is_noise" not in topic_lookup.columns:
            cols_to_merge.append("label_is_noise")
        
        if len(cols_to_merge) > 1:  # More than just topic_id
            topic_lookup = topic_lookup.merge(
                full_model_data[cols_to_merge].drop_duplicates("topic_id"),
                on="topic_id",
                how="left"
            )
            print(f"  ✓ Merged columns: {[c for c in cols_to_merge if c != 'topic_id']}")
    else:
        print("\n⚠️  full_model_data.parquet not found. Checking topic_lookup for keywords...")
        # Check if keywords exist directly in topic_lookup
        if "keywords" in topic_lookup.columns:
            print("  ✓ keywords found in topic_lookup")
        else:
            print("  ⚠️  keywords column not found in topic_lookup")
        print("  Available columns in topic_lookup:", list(topic_lookup.columns))

print("\nShapes:")
print(f"  book_wide: {book_wide.shape}")
print(f"  book_long: {book_long.shape}")
print(f"  book_topic_probs: {book_topic_probs.shape}")
print(f"  chapter_topic_probs: {chapter_topic_probs.shape}")
print(f"  topic_lookup: {topic_lookup.shape}")

# Merge labels into topic probabilities (for use in statistical analysis)
print("\nMerging labels from topic_lookup into book_topic_probs...")
book_topic_probs = book_topic_probs.merge(
    topic_lookup[["topic_id", "label"]].drop_duplicates("topic_id"),
    on="topic_id",
    how="left"
)

# Check label coverage
n_topics_with_labels = book_topic_probs["label"].notna().sum()
n_total = len(book_topic_probs)
print(f"  Topics with labels: {n_topics_with_labels:,} / {n_total:,} ({n_topics_with_labels/n_total*100:.1f}%)")
if book_topic_probs["label"].isna().any():
    missing_topics = book_topic_probs[book_topic_probs["label"].isna()]["topic_id"].unique()
    print(f"  ⚠️  {len(missing_topics)} topics missing labels: {sorted(missing_topics)[:10]}...")


✓ Loaded topic_lookup (REQUIRED for labels)

Shapes:
  book_wide: (92, 29)
  book_long: (2215, 22)
  book_topic_probs: (33856, 3)
  chapter_topic_probs: (1089280, 4)
  topic_lookup: (369, 21)

Merging labels from topic_lookup into book_topic_probs...
  Topics with labels: 33,856 / 33,856 (100.0%)


In [6]:
# 5b. Handle Missing Probability Mass at Segment Level (CRITICAL for H6)
# Account for missing probability mass per (book, segment) to prevent misleading arc analysis
# This must be done BEFORE any time-course or segment-level analysis

print("=" * 80)
print("Handling Missing Probability Mass at Segment Level:")
print("=" * 80)
print("\nCRITICAL: Segment sums were as low as 0.31. This must be corrected")
print("before any H6 time-course analysis to avoid misleading results.")
print("\nChanges across begin/middle/end might partially reflect coverage")
print("differences, not actual thematic changes, if missing mass is not handled.\n")

# Handle chapter-level missing mass
if {"book_id", "chapter_id"}.issubset(chapter_topic_probs.columns):
    print("Processing chapter-level segments...")
    
    # Compute probability sums per (book, chapter)
    chapter_prob_sums = chapter_topic_probs.groupby(["book_id", "chapter_id"])["prob"].sum()
    missing_mass_per_chapter = 1.0 - chapter_prob_sums
    
    print(f"\nChapter-level probability mass analysis:")
    print(f"  Mean sum per chapter: {chapter_prob_sums.mean():.4f}")
    print(f"  Min sum per chapter: {chapter_prob_sums.min():.4f}")
    print(f"  Max sum per chapter: {chapter_prob_sums.max():.4f}")
    print(f"  Mean missing mass per chapter: {missing_mass_per_chapter.mean():.4f}")
    print(f"  Total missing mass: {missing_mass_per_chapter.sum():.4f}")
    
    # Create OTHER bucket rows for missing mass per (book, chapter)
    other_chapter_rows = []
    for (book_id, chapter_id), missing in missing_mass_per_chapter.items():
        if missing > 1e-6:  # Only add if missing mass is meaningful
            other_chapter_rows.append({
                "book_id": book_id,
                "chapter_id": chapter_id,
                "topic_id": -1,  # Standard noise topic ID
                "prob": missing,
                "label": "OTHER (Missing/Noise Mass)"
            })
    
    if len(other_chapter_rows) > 0:
        other_chapter_df = pd.DataFrame(other_chapter_rows)
        print(f"\n  Created {len(other_chapter_rows)} OTHER bucket rows for chapters")
        print(f"  Total OTHER mass: {other_chapter_df['prob'].sum():.4f}")
        
        # Append to chapter_topic_probs
        chapter_topic_probs = pd.concat([chapter_topic_probs, other_chapter_df], ignore_index=True)
        
        # Verify: check new probability sums
        new_chapter_sums = chapter_topic_probs.groupby(["book_id", "chapter_id"])["prob"].sum()
        print(f"\n  After adding OTHER bucket:")
        print(f"    Mean sum per chapter: {new_chapter_sums.mean():.4f}")
        print(f"    Min sum per chapter: {new_chapter_sums.min():.4f}")
        print(f"    Max sum per chapter: {new_chapter_sums.max():.4f}")
        print(f"    Chapters with sum > 0.99: {(new_chapter_sums > 0.99).sum()} / {len(new_chapter_sums)}")
        print(f"    Chapters with sum > 1.01: {(new_chapter_sums > 1.01).sum()} / {len(new_chapter_sums)}")
        
        # ---- Begin Fix for KeyError: check if 'rating_tier' exists ----
        # Check if 'rating_tier' is present in book_wide before merging
        if "rating_tier" in book_wide.columns:
            # First, merge rating_tier onto chapter data
            chapter_topic_probs = chapter_topic_probs.merge(
                book_wide[["book_id", "rating_tier"]],
                on="book_id",
                how="left"
            )

            # Analyze OTHER by rating tier
            other_by_tier = chapter_topic_probs[
                chapter_topic_probs["topic_id"] == -1
            ].groupby("rating_tier")["prob"].mean()

            print(f"\n  OTHER mass by rating tier:")
            for tier, mass in other_by_tier.items():
                print(f"    {tier}: {mass:.4f}")
        else:
            print("\n  ⚠️  Skipping rating tier analysis: 'rating_tier' not found in book_wide columns.")
            print(f"  Columns available in book_wide: {book_wide.columns.tolist()}")
        # ---- End Fix for KeyError ----

        print(f"\n  ✓ Chapter-level OTHER bucket added and verified")
    else:
        print("\n  ⚠️  No missing mass detected at chapter level")
else:
    print("⚠️  Chapter-level data structure not available. Skipping chapter-level OTHER bucket.")

# Note: If tertile_topic_probs exists, handle it similarly
# (This would be done in a separate analysis notebook for H6)
print("\n" + "=" * 80)
print("NOTE: For tertile-level analysis (H6), apply the same OTHER bucket")
print("approach per (book, tertile) before any time-course analysis.")
print("=" * 80)


Handling Missing Probability Mass at Segment Level:

CRITICAL: Segment sums were as low as 0.31. This must be corrected
before any H6 time-course analysis to avoid misleading results.

Changes across begin/middle/end might partially reflect coverage
differences, not actual thematic changes, if missing mass is not handled.

Processing chapter-level segments...

Chapter-level probability mass analysis:
  Mean sum per chapter: 1.0000
  Min sum per chapter: 1.0000
  Max sum per chapter: 1.0000
  Mean missing mass per chapter: 0.0000
  Total missing mass: 0.0000

  ⚠️  No missing mass detected at chapter level

NOTE: For tertile-level analysis (H6), apply the same OTHER bucket
approach per (book, tertile) before any time-course analysis.


In [7]:
# 3b. Fix data issues: book_id type mismatch, load rating_class, check NaN probabilities

print("=" * 80)
print("Fixing data issues:")
print("=" * 80)

# 1. Fix book_id type mismatch: convert book_topic_probs book_id from string to int
print("\n1. Fixing book_id type mismatch...")
print(f"  book_wide['book_id'].dtype: {book_wide['book_id'].dtype}")
print(f"  book_topic_probs['book_id'].dtype: {book_topic_probs['book_id'].dtype}")

# Convert book_topic_probs book_id to numeric (matching book_wide)
book_topic_probs['book_id'] = pd.to_numeric(book_topic_probs['book_id'], errors='coerce')
print(f"  After conversion - book_topic_probs['book_id'].dtype: {book_topic_probs['book_id'].dtype}")

# Check for any conversion failures
failed_conversions = book_topic_probs['book_id'].isna().sum()
if failed_conversions > 0:
    print(f"  ⚠️  Warning: {failed_conversions} book_ids failed to convert to numeric")

# 2. Check NaN probabilities
print("\n2. Checking NaN probabilities...")
nan_probs = book_topic_probs['prob'].isna().sum()
total_probs = len(book_topic_probs)
print(f"  NaN probabilities: {nan_probs:,} / {total_probs:,} ({nan_probs/total_probs*100:.1f}%)")

if nan_probs > 0:
    print("  ⚠️  Warning: Found NaN probabilities. Checking which topics/books are affected...")
    nan_by_topic = book_topic_probs[book_topic_probs['prob'].isna()].groupby('topic_id').size().sort_values(ascending=False)
    nan_by_book = book_topic_probs[book_topic_probs['prob'].isna()].groupby('book_id').size().sort_values(ascending=False)
    print(f"  Topics with NaN: {len(nan_by_topic)} unique topics")
    print(f"  Books with NaN: {len(nan_by_book)} unique books")
    print(f"  Sample topics with NaN: {nan_by_topic.head(10).to_dict()}")
    
    # Drop rows with NaN probabilities for now (we'll investigate the root cause separately)
    print("  Dropping rows with NaN probabilities...")
    book_topic_probs = book_topic_probs.dropna(subset=['prob'])
    print(f"  After dropping NaN: {len(book_topic_probs):,} rows remaining")

# 3. Load goodreads.csv and create rating_class
print("\n3. Loading goodreads.csv and creating rating_class...")
if not GOODREADS_PATH.exists():
    raise FileNotFoundError(f"goodreads.csv not found at {GOODREADS_PATH}")

goodreads = pd.read_csv(GOODREADS_PATH)
print(f"  Loaded goodreads.csv: {goodreads.shape}")
print(f"  Columns: {list(goodreads.columns)}")

# Standardize book_id column name (could be ID, book_id, etc.)
if 'ID' in goodreads.columns:
    goodreads = goodreads.rename(columns={'ID': 'book_id'})
elif 'goodreads_book_id' in goodreads.columns:
    goodreads = goodreads.rename(columns={'goodreads_book_id': 'book_id'})

# Convert book_id to numeric to match book_wide
goodreads['book_id'] = pd.to_numeric(goodreads['book_id'], errors='coerce')

# Check if Score column exists (this is the rating)
if 'Score' not in goodreads.columns:
    print(f"  ⚠️  Warning: 'Score' column not found. Available columns: {list(goodreads.columns)}")
    # Try alternative names
    if 'rating_mean' in goodreads.columns:
        goodreads = goodreads.rename(columns={'rating_mean': 'Score'})
    elif 'avg_rating' in goodreads.columns:
        goodreads = goodreads.rename(columns={'avg_rating': 'Score'})
    else:
        raise KeyError(f"Could not find rating column. Available: {list(goodreads.columns)}")

# Create rating_class using quantiles (0.33, 0.66) as per methodology
print("  Creating rating_class based on Score quantiles...")
book_ratings = goodreads['Score'].dropna()
low_q, high_q = book_ratings.quantile([0.33, 0.66])
print(f"  Quantiles: {low_q:.3f} (33rd), {high_q:.3f} (66th)")

def assign_rating_class(score):
    if pd.isna(score):
        return None
    if score < low_q:
        return "bad"
    elif score <= high_q:
        return "mid"
    else:
        return "good"

goodreads['rating_class'] = goodreads['Score'].apply(assign_rating_class)
print(f"  Rating class distribution:")
print(goodreads['rating_class'].value_counts())

# 4. Merge rating_class into book_wide
print("\n4. Merging rating_class into book_wide...")
book_wide = book_wide.merge(
    goodreads[['book_id', 'rating_class', 'Score']],
    on='book_id',
    how='left'
)
print(f"  After merge: {book_wide.shape}")
print(f"  Books with rating_class: {book_wide['rating_class'].notna().sum()} / {len(book_wide)}")
print(f"  Rating class distribution in book_wide:")
print(book_wide['rating_class'].value_counts(dropna=False))

print("\n✓ Data fixes completed!")

Fixing data issues:

1. Fixing book_id type mismatch...
  book_wide['book_id'].dtype: Int64
  book_topic_probs['book_id'].dtype: object
  After conversion - book_topic_probs['book_id'].dtype: int64

2. Checking NaN probabilities...
  NaN probabilities: 0 / 33,856 (0.0%)

3. Loading goodreads.csv and creating rating_class...
  Loaded goodreads.csv: (97, 14)
  Columns: ['ID', 'Author', 'Title', 'URL', 'SeriesName', 'Summary', 'Genres', 'Score', 'RatingsCount', 'ReviewsCount', 'Pages', 'PublishedDate', 'Popularity_ReadingNow', 'Popularity_Wishlisted']
  Creating rating_class based on Score quantiles...
  Quantiles: 3.917 (33rd), 4.074 (66th)
  Rating class distribution:
rating_class
good    33
mid     32
bad     32
Name: count, dtype: int64

4. Merging rating_class into book_wide...
  After merge: (92, 31)
  Books with rating_class: 92 / 92
  Rating class distribution in book_wide:
rating_class
mid     32
good    30
bad     30
Name: count, dtype: int64

✓ Data fixes completed!


In [8]:
# 4. Integrity checks
n_topics_expected = 368

print("=" * 80)
print("Integrity checks:")
print("=" * 80)

# Check topics count
print("Unique topics (book):", book_topic_probs["topic_id"].nunique())
print("Unique topics (chapter):", chapter_topic_probs["topic_id"].nunique())

# Probability sums per book
book_sums = book_topic_probs.groupby("book_id")["prob"].sum()
print("Book prob sums — min/max:", book_sums.min(), book_sums.max())

# Probability sums per (book, chapter)
if {"book_id", "chapter_id"}.issubset(chapter_topic_probs.columns):
    chapter_sums = chapter_topic_probs.groupby(["book_id", "chapter_id"])["prob"].sum()
    print("Chapter prob sums — min/max:", chapter_sums.min(), chapter_sums.max())

# Basic cohort overlap (should now work after fixing book_id types)
book_ids_probs = set(book_topic_probs["book_id"].dropna().unique())
book_ids_wide = set(book_wide["book_id"].dropna().unique())
overlap = book_ids_probs & book_ids_wide
print("Book IDs overlap (topic_probs ∩ wide):", len(overlap), "of", len(book_ids_probs))
if len(overlap) == 0:
    print("  ⚠️  WARNING: No overlap! Check book_id formats:")
    print(f"  book_ids_probs sample: {sorted(list(book_ids_probs))[:5]}")
    print(f"  book_ids_wide sample: {sorted(list(book_ids_wide))[:5]}")
    print(f"  book_ids_probs dtype: {book_topic_probs['book_id'].dtype}")
    print(f"  book_ids_wide dtype: {book_wide['book_id'].dtype}")

# Rating class availability (should now be present after merge)
if "rating_class" in book_wide.columns:
    print("✓ Rating classes present:", sorted(book_wide["rating_class"].dropna().unique()))
    print("  Rating class distribution:")
    print(book_wide["rating_class"].value_counts(dropna=False))
else:
    print("⚠️ rating_class column missing in book_wide")

Integrity checks:
Unique topics (book): 368
Unique topics (chapter): 369
Book prob sums — min/max: 0.6201516892595613 0.7312169573083945
Chapter prob sums — min/max: 1.0 1.0
Book IDs overlap (topic_probs ∩ wide): 92 of 92
✓ Rating classes present: ['bad', 'good', 'mid']
  Rating class distribution:
rating_class
mid     32
good    30
bad     30
Name: count, dtype: int64


In [9]:
# 6. Rating-tier prep (Top / Middle / Trash)
# rating_class should already be in book_wide from the previous cell
print("=" * 80)
print("Rating-tier preparation:")
print("=" * 80)

# Expect rating_class in {"good","mid","bad"}; map to desired labels
rating_map = {
    "good": "top",
    "bad": "trash",
    "mid": "middle",
    "top": "top",
    "trash": "trash",
    "middle": "middle",
}

if "rating_class" not in book_wide.columns:
    print("⚠️ rating_class not found in book_wide.columns")
    print("Available columns:", list(book_wide.columns))
    raise KeyError("rating_class not found in book_wide; please check the data loading and merge steps")

# Check what values we have
print("rating_class values in book_wide:", sorted(book_wide["rating_class"].dropna().unique()))

book_wide = book_wide.copy()
book_wide["rating_tier"] = book_wide["rating_class"].map(rating_map)

# Check for unmapped values
unmapped = book_wide[book_wide["rating_tier"].isna() & book_wide["rating_class"].notna()]
if len(unmapped) > 0:
    print(f"⚠️  Warning: {len(unmapped)} books with unmapped rating_class values:")
    print(unmapped["rating_class"].value_counts())

# Merge rating_tier onto topic probabilities
print("\nMerging rating_tier onto topic probabilities...")
book_topic_probs = book_topic_probs.merge(
    book_wide[["book_id", "rating_tier"]], on="book_id", how="left"
)

# Check merge success
merged_count = book_topic_probs["rating_tier"].notna().sum()
total_count = len(book_topic_probs)
print(f"  Merged rating_tier: {merged_count:,} / {total_count:,} ({merged_count/total_count*100:.1f}%)")

# Use all three tiers by default
book_topic_probs_all = book_topic_probs.copy()

# Optional helper: Top vs Trash subset for pairwise contrasts if needed later
book_topic_probs_top_trash = book_topic_probs_all[book_topic_probs_all["rating_tier"].isin(["top", "trash"])].copy()

print("\nBooks per tier (all):")
print(book_topic_probs_all.groupby("rating_tier")["book_id"].nunique())
print("\nBooks per tier (Top/Trash subset, optional):")
print(book_topic_probs_top_trash.groupby("rating_tier")["book_id"].nunique())

Rating-tier preparation:
rating_class values in book_wide: ['bad', 'good', 'mid']

Merging rating_tier onto topic probabilities...
  Merged rating_tier: 33,856 / 33,856 (100.0%)

Books per tier (all):
rating_tier
middle    32
top       30
trash     30
Name: book_id, dtype: int64

Books per tier (Top/Trash subset, optional):
rating_tier
top      30
trash    30
Name: book_id, dtype: int64


In [10]:
# 5. Handle Missing Probability Mass (after rating-tier prep)
# Account for the ~0.38 missing probability mass by creating an "OTHER" bucket
# This represents unmodeled topics/noise that BERTopic didn't capture

print("=" * 80)
print("Handling Missing Probability Mass:")
print("=" * 80)

# Compute probability sums per book
book_prob_sums = book_topic_probs_all.groupby("book_id")["prob"].sum()
missing_mass_per_book = 1.0 - book_prob_sums

print(f"\nProbability mass analysis:")
print(f"  Mean sum per book: {book_prob_sums.mean():.4f}")
print(f"  Min sum per book: {book_prob_sums.min():.4f}")
print(f"  Max sum per book: {book_prob_sums.max():.4f}")
print(f"  Mean missing mass per book: {missing_mass_per_book.mean():.4f}")
print(f"  Total missing mass: {missing_mass_per_book.sum():.4f}")

# Create "OTHER" bucket rows for missing mass
# We'll add one row per book with topic_id=-1 (standard noise topic ID)
other_rows = []
for book_id in book_prob_sums.index:
    missing = missing_mass_per_book[book_id]
    if missing > 1e-6:  # Only add if missing mass is meaningful
        # Get rating_tier for this book
        book_tier = book_topic_probs_all[book_topic_probs_all["book_id"] == book_id]["rating_tier"].iloc[0] if len(book_topic_probs_all[book_topic_probs_all["book_id"] == book_id]) > 0 else None
        if book_tier is not None:
            other_rows.append({
                "book_id": book_id,
                "topic_id": -1,  # Standard noise topic ID
                "prob": missing,
                "label": "OTHER (Missing/Noise Mass)",
                "rating_tier": book_tier
            })

if len(other_rows) > 0:
    other_df = pd.DataFrame(other_rows)
    print(f"\n  Created {len(other_rows)} OTHER bucket rows")
    print(f"  Total OTHER mass: {other_df['prob'].sum():.4f}")
    
    # Append to book_topic_probs_all
    book_topic_probs_all = pd.concat([book_topic_probs_all, other_df], ignore_index=True)
    
    # Verify: check new probability sums
    new_book_sums = book_topic_probs_all.groupby("book_id")["prob"].sum()
    print(f"\n  After adding OTHER bucket:")
    print(f"    Mean sum per book: {new_book_sums.mean():.4f}")
    print(f"    Min sum per book: {new_book_sums.min():.4f}")
    print(f"    Max sum per book: {new_book_sums.max():.4f}")
    print(f"    Books with sum > 0.99: {(new_book_sums > 0.99).sum()} / {len(new_book_sums)}")
    print(f"    Books with sum > 1.01: {(new_book_sums > 1.01).sum()} / {len(new_book_sums)}")
    
    # Add OTHER to topic_lookup if not present
    if -1 not in topic_lookup["topic_id"].values:
        other_lookup = pd.DataFrame([{
            "topic_id": -1,
            "label": "OTHER (Missing/Noise Mass)",
            "keywords": "unmodeled topics, noise, missing mass",
            "scene_summary": "Represents probability mass not captured by the 368 BERTopic topics"
        }])
        topic_lookup = pd.concat([topic_lookup, other_lookup], ignore_index=True)
        print(f"\n  ✓ Added OTHER to topic_lookup")
else:
    print("\n  ⚠️  No missing mass detected or all books have complete probability coverage")

print("\n✓ Missing probability mass handling completed")


Handling Missing Probability Mass:

Probability mass analysis:
  Mean sum per book: 0.6802
  Min sum per book: 0.6202
  Max sum per book: 0.7312
  Mean missing mass per book: 0.3198
  Total missing mass: 29.4237

  Created 92 OTHER bucket rows
  Total OTHER mass: 29.4237

  After adding OTHER bucket:
    Mean sum per book: 1.0000
    Min sum per book: 1.0000
    Max sum per book: 1.0000
    Books with sum > 0.99: 92 / 92
    Books with sum > 1.01: 0 / 92

✓ Missing probability mass handling completed


In [11]:
# 7. Helper utilities

def show_plotly_fig(fig, save_html=True, output_dir=FIG_DIR):
    """Display Plotly figure; optionally save to HTML."""
    try:
        fig.show()
    except (ValueError, ImportError):
        pass
    if save_html and output_dir is not None:
        output_dir.mkdir(parents=True, exist_ok=True)
        html_path = output_dir / f"plot_{hash(str(fig.layout.title.text if fig.layout.title else 'figure'))}.html"
        fig.write_html(str(html_path))
        print(f"Saved interactive plot to: {html_path}")


def cliffs_delta(x: np.ndarray, y: np.ndarray) -> float:
    """Cliff's delta effect size."""
    x = np.asarray(x)
    y = np.asarray(y)
    gt = np.sum(x[:, None] > y[None, :])
    lt = np.sum(x[:, None] < y[None, :])
    return (gt - lt) / (len(x) * len(y))


def compute_topic_health(df: pd.DataFrame, threshold: float = 0.001) -> pd.DataFrame:
    """Compute prevalence/mass/concentration per topic using LABELS (not topic_id)."""
    # Group by label instead of topic_id for outputs
    prevalence = (df["prob"] > threshold).groupby(df["label"]).mean().rename("prevalence")
    mass = df.groupby("label")["prob"].mean().rename("mass")
    # simple concentration proxy: max share / mean share per topic
    concentration = (
        df.groupby("label")["prob"].max() / df.groupby("label")["prob"].mean().replace(0, np.nan)
    ).rename("concentration_ratio")
    # Also keep topic_id for reference
    topic_id_map = df.groupby("label")["topic_id"].first().rename("topic_id")
    out = pd.concat([prevalence, mass, concentration, topic_id_map], axis=1).reset_index()
    return out

In [12]:
# 8. Topic health metrics (prevalence, mass, concentration)
# Using labels (not topic_id) for outputs as per requirements

print("=" * 80)
print("Computing topic health metrics:")
print("=" * 80)

topic_health = compute_topic_health(book_topic_probs_all, threshold=0.001)
print(f"\nTopic health computed for {len(topic_health)} topics")

# Merge with topic_lookup to add keywords and interpretation commentary
lookup_cols = ["label"]
if "keywords" in topic_lookup.columns:
    lookup_cols.append("keywords")
if "label_rationale" in topic_lookup.columns:
    lookup_cols.append("label_rationale")
if "scene_summary" in topic_lookup.columns:
    lookup_cols.append("scene_summary")
if "label_is_noise" in topic_lookup.columns:
    lookup_cols.append("label_is_noise")

# Merge using topic_id from topic_health
topic_health = topic_health.merge(
    topic_lookup[["topic_id"] + [c for c in lookup_cols if c != "label"]].drop_duplicates("topic_id"),
    on="topic_id",
    how="left"
)

print(f"\nTopic health summary:")
print(topic_health.describe())

# Save topic health table
topic_health_path = TABLE_DIR / "topic_health_table.parquet"
topic_health.to_parquet(topic_health_path, index=False)
topic_health_csv_path = TABLE_DIR / "topic_health_table.csv"
topic_health.to_csv(topic_health_csv_path, index=False)
print(f"\n✓ Saved topic health table to: {topic_health_path}")
print(f"✓ Saved topic health table (CSV) to: {topic_health_csv_path}")

# Show top topics by prevalence
print("\nTop 10 topics by prevalence:")
print(topic_health.nlargest(10, "prevalence")[["label", "prevalence", "mass", "concentration_ratio"]])

print("\nTop 10 topics by mass:")
print(topic_health.nlargest(10, "mass")[["label", "prevalence", "mass", "concentration_ratio"]])


Computing topic health metrics:

Topic health computed for 343 topics

Topic health summary:
       prevalence        mass  concentration_ratio    topic_id
count  343.000000  343.000000           343.000000  343.000000
mean     0.575538    0.002734             8.077546  184.819242
std      0.401927    0.017250            11.432865  106.688139
min      0.000000    0.000003             1.187682   -1.000000
25%      0.097826    0.000829             2.068178   93.500000
50%      0.728261    0.001368             2.663997  187.000000
75%      0.967391    0.002315             7.636642  274.500000
max      1.000000    0.319823            63.008440  367.000000

✓ Saved topic health table to: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/correlation_analysis/01_topic_analysis/tables/topic_health_table.parquet
✓ Saved topic health table (CSV) to: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/resu

In [13]:
# 9. Visual Exploration (1.2): Distribution plots per topic
# Create summary visualizations and detailed plots for top N topics

print("=" * 80)
print("Visual Exploration: Distribution plots")
print("=" * 80)

# Create subdirectories for figures
DIST_FIG_DIR = FIG_DIR / "topic_distributions"
DIST_FIG_DIR.mkdir(parents=True, exist_ok=True)
TOP_N_FIG_DIR = DIST_FIG_DIR / "top_20_topics_detailed"
TOP_N_FIG_DIR.mkdir(parents=True, exist_ok=True)

# Get top 20 topics by prevalence for detailed plots
top_n = 20
top_topics = topic_health.nlargest(top_n, "prevalence")["label"].tolist()
print(f"\nSelected top {top_n} topics by prevalence for detailed visualization:")
for i, label in enumerate(top_topics[:10], 1):
    print(f"  {i}. {label}")
if len(top_topics) > 10:
    print(f"  ... and {len(top_topics) - 10} more")

def plot_topic_distribution_violin(topic_label: str, df: pd.DataFrame, save_path: Path):
    """Create violin plot for a single topic across rating tiers."""
    topic_data = df[df["label"] == topic_label].copy()
    
    if len(topic_data) == 0:
        print(f"  ⚠️  No data for topic: {topic_label}")
        return None
    
    fig = go.Figure()
    
    for tier in ["top", "middle", "trash"]:
        tier_data = topic_data[topic_data["rating_tier"] == tier]["prob"]
        if len(tier_data) > 0:
            fig.add_trace(go.Violin(
                y=tier_data,
                name=tier.capitalize(),
                box_visible=True,
                meanline_visible=True,
                fillcolor=f"rgba({['31, 119, 180', '255, 127, 14', '44, 160, 44'][['top', 'middle', 'trash'].index(tier)]}, 0.6)",
                line_color=f"rgb({['31, 119, 180', '255, 127, 14', '44, 160, 44'][['top', 'middle', 'trash'].index(tier)]})",
                opacity=0.7
            ))
    
    fig.update_layout(
        title=f"Topic Distribution: {topic_label}",
        yaxis_title="Probability",
        xaxis_title="Rating Tier",
        violinmode="group",
        height=500,
        showlegend=True
    )
    
    # Save
    safe_label = "".join(c if c.isalnum() or c in (' ', '-', '_') else '_' for c in topic_label)[:50]
    html_path = save_path / f"violin_{safe_label}.html"
    fig.write_html(str(html_path))
    
    return fig

def plot_topic_distribution_box(topic_label: str, df: pd.DataFrame, save_path: Path):
    """Create box plot with jitter for a single topic."""
    topic_data = df[df["label"] == topic_label].copy()
    
    if len(topic_data) == 0:
        return None
    
    fig = go.Figure()
    
    for tier in ["top", "middle", "trash"]:
        tier_data = topic_data[topic_data["rating_tier"] == tier]
        if len(tier_data) > 0:
            # Add jitter
            jittered_x = np.random.normal(
                ["top", "middle", "trash"].index(tier),
                0.05,
                len(tier_data)
            )
            
            fig.add_trace(go.Scatter(
                x=jittered_x,
                y=tier_data["prob"],
                mode="markers",
                name=tier.capitalize(),
                marker=dict(
                    size=4,
                    opacity=0.5,
                    color=f"rgb({['31, 119, 180', '255, 127, 14', '44, 160, 44'][['top', 'middle', 'trash'].index(tier)]})"
                ),
                showlegend=False
            ))
            
            # Add box plot
            fig.add_trace(go.Box(
                y=tier_data["prob"],
                name=tier.capitalize(),
                boxmean="sd",
                fillcolor=f"rgba({['31, 119, 180', '255, 127, 14', '44, 160, 44'][['top', 'middle', 'trash'].index(tier)]}, 0.3)",
                line_color=f"rgb({['31, 119, 180', '255, 127, 14', '44, 160, 44'][['top', 'middle', 'trash'].index(tier)]})",
            ))
    
    fig.update_layout(
        title=f"Topic Distribution (Box + Jitter): {topic_label}",
        yaxis_title="Probability",
        xaxis_title="Rating Tier",
        xaxis=dict(tickmode="array", tickvals=[0, 1, 2], ticktext=["Top", "Middle", "Trash"]),
        height=500
    )
    
    safe_label = "".join(c if c.isalnum() or c in (' ', '-', '_') else '_' for c in topic_label)[:50]
    html_path = save_path / f"box_{safe_label}.html"
    fig.write_html(str(html_path))
    
    return fig

# Create detailed plots for top N topics
print(f"\nCreating detailed plots for top {top_n} topics...")
for i, label in enumerate(top_topics, 1):
    print(f"  [{i}/{top_n}] {label}")
    plot_topic_distribution_violin(label, book_topic_probs_all, TOP_N_FIG_DIR)
    plot_topic_distribution_box(label, book_topic_probs_all, TOP_N_FIG_DIR)

print(f"\n✓ Saved detailed plots to: {TOP_N_FIG_DIR}")


Visual Exploration: Distribution plots

Selected top 20 topics by prevalence for detailed visualization:
  1. Anger Management In Relationship
  2. Angry Argument
  3. Apologetic Excuse Me
  4. Arrogant Man's Behavior
  5. Bedroom Intimacy
  6. Belief Discussion
  7. Color Scheme Discussion
  8. Desperate Harem Offer
  9. Emotional Argument
  10. Emotional Delusion Conversation
  ... and 10 more

Creating detailed plots for top 20 topics...
  [1/20] Anger Management In Relationship
  [2/20] Angry Argument
  [3/20] Apologetic Excuse Me
  [4/20] Arrogant Man's Behavior
  [5/20] Bedroom Intimacy
  [6/20] Belief Discussion
  [7/20] Color Scheme Discussion
  [8/20] Desperate Harem Offer
  [9/20] Emotional Argument
  [10/20] Emotional Delusion Conversation
  [11/20] Emotional Love Confession
  [12/20] Epic Love Confessions
  [13/20] Firm Rejections
  [14/20] Friendship Bonding
  [15/20] Hatred-fueled Argument
  [16/20] Hushed Conversations
  [17/20] Indoor Pool Swimming
  [18/20] Knocking On

In [14]:
# 10. Create summary visualization: All topics grid (violin plots)
# Batch visualization for all topics in a grid format

print("=" * 80)
print("Creating summary visualization: All topics grid")
print("=" * 80)

# Sample a subset for grid visualization (all 368 would be too many)
# Show top 30 by prevalence + 10 random others for diversity
n_grid = 40
top_grid = topic_health.nlargest(30, "prevalence")["label"].tolist()
remaining = topic_health[~topic_health["label"].isin(top_grid)]["label"].tolist()
random_sample = np.random.choice(remaining, min(10, len(remaining)), replace=False).tolist()
grid_topics = top_grid + random_sample

print(f"Creating grid visualization for {len(grid_topics)} topics (top 30 + 10 random)")

# Create subplots
n_cols = 5
n_rows = int(np.ceil(len(grid_topics) / n_cols))
# Calculate adaptive vertical spacing (max allowed is 1/(rows-1))
max_vertical_spacing = 1.0 / (n_rows - 1) if n_rows > 1 else 0.3
vertical_spacing = min(0.15, max_vertical_spacing * 0.9)  # Use 90% of max to be safe

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=grid_topics,
    vertical_spacing=vertical_spacing,
    horizontal_spacing=0.1
)

for idx, label in enumerate(grid_topics):
    row = (idx // n_cols) + 1
    col = (idx % n_cols) + 1
    
    topic_data = book_topic_probs_all[book_topic_probs_all["label"] == label]
    
    for tier_idx, tier in enumerate(["top", "middle", "trash"]):
        tier_data = topic_data[topic_data["rating_tier"] == tier]["prob"]
        if len(tier_data) > 0:
            fig.add_trace(
                go.Violin(
                    y=tier_data,
                    name=tier,
                    box_visible=True,
                    meanline_visible=True,
                    showlegend=(idx == 0),  # Only show legend for first subplot
                    fillcolor=f"rgba({['31, 119, 180', '255, 127, 14', '44, 160, 44'][tier_idx]}, 0.6)",
                    line_color=f"rgb({['31, 119, 180', '255, 127, 14', '44, 160, 44'][tier_idx]})",
                    opacity=0.7
                ),
                row=row,
                col=col
            )

fig.update_layout(
    title="Topic Distributions Across Rating Tiers (Sample)",
    height=200 * n_rows,
    showlegend=True
)

# Save
grid_path = DIST_FIG_DIR / "all_topics_violin_grid.html"
fig.write_html(str(grid_path))
print(f"\n✓ Saved grid visualization to: {grid_path}")


Creating summary visualization: All topics grid
Creating grid visualization for 40 topics (top 30 + 10 random)

✓ Saved grid visualization to: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/correlation_analysis/01_topic_analysis/figures/topic_distributions/all_topics_violin_grid.html


In [15]:
# 11. Quantify Differences Per Topic (1.3)
# Compute group-wise statistics, effect sizes, and significance tests

print("=" * 80)
print("Quantifying differences per topic:")
print("=" * 80)

def compute_topic_statistics(df: pd.DataFrame) -> pd.DataFrame:
    """Compute comprehensive statistics for each topic across rating tiers."""
    results = []
    
    unique_labels = df["label"].dropna().unique()
    print(f"Computing statistics for {len(unique_labels)} topics...")
    
    for label in unique_labels:
        topic_data = df[df["label"] == label].copy()
        
        if len(topic_data) == 0:
            continue
        
        # Get topic_id (should be same for all rows with same label)
        topic_id = topic_data["topic_id"].iloc[0]
        
        # Group-wise statistics
        stats_by_tier = {}
        for tier in ["top", "middle", "trash"]:
            tier_data = topic_data[topic_data["rating_tier"] == tier]["prob"]
            
            if len(tier_data) == 0:
                stats_by_tier[tier] = {
                    "n": 0,
                    "mean": np.nan,
                    "median": np.nan,
                    "q1": np.nan,
                    "q3": np.nan,
                    "std": np.nan
                }
            else:
                stats_by_tier[tier] = {
                    "n": len(tier_data),
                    "mean": tier_data.mean(),
                    "median": tier_data.median(),
                    "q1": tier_data.quantile(0.25),
                    "q3": tier_data.quantile(0.75),
                    "std": tier_data.std()
                }
        
        # Effect sizes (Top vs Trash, primary comparison)
        top_probs = topic_data[topic_data["rating_tier"] == "top"]["prob"].values
        trash_probs = topic_data[topic_data["rating_tier"] == "trash"]["prob"].values
        middle_probs = topic_data[topic_data["rating_tier"] == "middle"]["prob"].values
        
        cliffs_top_trash = np.nan
        cliffs_top_middle = np.nan
        cliffs_middle_trash = np.nan
        
        if len(top_probs) > 0 and len(trash_probs) > 0:
            cliffs_top_trash = cliffs_delta(top_probs, trash_probs)
        
        if len(top_probs) > 0 and len(middle_probs) > 0:
            cliffs_top_middle = cliffs_delta(top_probs, middle_probs)
        
        if len(middle_probs) > 0 and len(trash_probs) > 0:
            cliffs_middle_trash = cliffs_delta(middle_probs, trash_probs)
        
        # Significance tests
        # Kruskal-Wallis for 3 groups
        all_tiers = []
        tier_labels = []
        for tier in ["top", "middle", "trash"]:
            tier_data = topic_data[topic_data["rating_tier"] == tier]["prob"].values
            if len(tier_data) > 0:
                all_tiers.append(tier_data)
                tier_labels.extend([tier] * len(tier_data))
        
        kruskal_stat = np.nan
        kruskal_p = np.nan
        if len(all_tiers) >= 2:
            try:
                kruskal_stat, kruskal_p = kruskal(*all_tiers)
            except:
                pass
        
        # Mann-Whitney U for pairwise comparisons
        mw_top_trash_stat = np.nan
        mw_top_trash_p = np.nan
        if len(top_probs) > 0 and len(trash_probs) > 0:
            try:
                mw_top_trash_stat, mw_top_trash_p = mannwhitneyu(top_probs, trash_probs, alternative="two-sided")
            except:
                pass
        
        mw_top_middle_stat = np.nan
        mw_top_middle_p = np.nan
        if len(top_probs) > 0 and len(middle_probs) > 0:
            try:
                mw_top_middle_stat, mw_top_middle_p = mannwhitneyu(top_probs, middle_probs, alternative="two-sided")
            except:
                pass
        
        mw_middle_trash_stat = np.nan
        mw_middle_trash_p = np.nan
        if len(middle_probs) > 0 and len(trash_probs) > 0:
            try:
                mw_middle_trash_stat, mw_middle_trash_p = mannwhitneyu(middle_probs, trash_probs, alternative="two-sided")
            except:
                pass
        
        # Combine results
        result = {
            "topic_id": topic_id,
            "label": label,
            # Top tier stats
            "top_n": stats_by_tier["top"]["n"],
            "top_mean": stats_by_tier["top"]["mean"],
            "top_median": stats_by_tier["top"]["median"],
            "top_q1": stats_by_tier["top"]["q1"],
            "top_q3": stats_by_tier["top"]["q3"],
            "top_std": stats_by_tier["top"]["std"],
            # Middle tier stats
            "middle_n": stats_by_tier["middle"]["n"],
            "middle_mean": stats_by_tier["middle"]["mean"],
            "middle_median": stats_by_tier["middle"]["median"],
            "middle_q1": stats_by_tier["middle"]["q1"],
            "middle_q3": stats_by_tier["middle"]["q3"],
            "middle_std": stats_by_tier["middle"]["std"],
            # Trash tier stats
            "trash_n": stats_by_tier["trash"]["n"],
            "trash_mean": stats_by_tier["trash"]["mean"],
            "trash_median": stats_by_tier["trash"]["median"],
            "trash_q1": stats_by_tier["trash"]["q1"],
            "trash_q3": stats_by_tier["trash"]["q3"],
            "trash_std": stats_by_tier["trash"]["std"],
            # Effect sizes
            "cliffs_top_trash": cliffs_top_trash,
            "cliffs_top_middle": cliffs_top_middle,
            "cliffs_middle_trash": cliffs_middle_trash,
            # Significance tests
            "kruskal_stat": kruskal_stat,
            "kruskal_p": kruskal_p,
            "mw_top_trash_stat": mw_top_trash_stat,
            "mw_top_trash_p": mw_top_trash_p,
            "mw_top_middle_stat": mw_top_middle_stat,
            "mw_top_middle_p": mw_top_middle_p,
            "mw_middle_trash_stat": mw_middle_trash_stat,
            "mw_middle_trash_p": mw_middle_trash_p,
        }
        
        results.append(result)
    
    return pd.DataFrame(results)

# Compute statistics
topic_stats = compute_topic_statistics(book_topic_probs_all)

# Apply FDR correction to p-values
print("\nApplying FDR correction to p-values...")
p_values = topic_stats[["kruskal_p", "mw_top_trash_p", "mw_top_middle_p", "mw_middle_trash_p"]].values.flatten()
p_values = p_values[~np.isnan(p_values)]

if len(p_values) > 0:
    _, p_adjusted, _, _ = multipletests(p_values, method="fdr_bh", alpha=0.05)
    
    # Map back to original structure (this is approximate - full implementation would track indices)
    # For now, we'll do FDR correction per test type
    for col in ["kruskal_p", "mw_top_trash_p", "mw_top_middle_p", "mw_middle_trash_p"]:
        p_vals = topic_stats[col].values
        valid_mask = ~np.isnan(p_vals)
        if valid_mask.sum() > 0:
            _, p_adj, _, _ = multipletests(p_vals[valid_mask], method="fdr_bh", alpha=0.05)
            topic_stats[f"{col}_adj"] = np.nan
            topic_stats.loc[valid_mask, f"{col}_adj"] = p_adj

print(f"\n✓ Computed statistics for {len(topic_stats)} topics")
print(f"\nSample statistics:")
print(topic_stats[["label", "top_median", "trash_median", "cliffs_top_trash", "kruskal_p"]].head(10))


Quantifying differences per topic:
Computing statistics for 343 topics...

Applying FDR correction to p-values...

✓ Computed statistics for 343 topics

Sample statistics:
                                  label  top_median  trash_median  \
0                      Negotiating Deal    0.000563      0.000573   
1    Intimate Breast And Nipple Kissing    0.003126      0.004122   
2  Clitoral Stimulation During Foreplay    0.005653      0.004120   
3                     Dinner Invitation    0.003639      0.003525   
4         Unclear Relationship Feelings    0.001887      0.001991   
5      Long-term Relationship Struggles    0.002195      0.002245   
6                Wine-drinking At Table    0.005224      0.004568   
7   Relationship Ambiguity Conversation    0.011618      0.011547   
8            Marriage Ceremony Planning    0.003760      0.002217   
9                      Knocking On Door    0.005003      0.004065   

   cliffs_top_trash  kruskal_p  
0          0.053333   0.794189  
1 

In [17]:
# 15. Topic-to-Hypothesis Mapping & Procedure Topics Classification
# Build a shortlist of topics that map to specific hypotheses (full filtered set, not just top hits)
# Bridge to the subgroup analysis stage

print("=" * 80)
print("Topic-to-Hypothesis Mapping & Procedure Topics Classification")
print("=" * 80)

# Robust: try to get ALL Tier 2 topics; if not available, fallback to basic gate filtering
filtered_topics = None
if "tier2" in globals() and tier2 is not None and len(tier2) > 0:
    filtered_topics = tier2.copy()
    n_tier1 = len(tier1) if "tier1" in globals() and tier1 is not None else 0
    print(f"\nUsing ALL Tier 2 topics: {len(filtered_topics)} topics (includes {n_tier1} from Tier 1)")
else:
    # Defensive: check that topic_leaderboard_all exists and is not empty
    if "topic_leaderboard_all" in globals() and topic_leaderboard_all is not None and len(topic_leaderboard_all) > 0:
        filtered_topics = topic_leaderboard_all[
            (topic_leaderboard_all["abs_cliffs_top_trash"] >= 0.20) &
            (topic_leaderboard_all["prevalence"] >= 0.10)
        ].copy()
        print(f"\nFallback: Using filtered topics from topic_leaderboard_all: {len(filtered_topics)} topics")
    else:
        raise RuntimeError("No Tier 2 topics or topic_leaderboard_all available for topic-to-hypothesis mapping.")

print(f"\nMapping {len(filtered_topics)} filtered topics to hypotheses...")

# Hypothesis definitions: each is a (keywords, taxonomies) tuple for patterning
hypothesis_definitions = {
    "H1_Explicit": (
        ["sexual", "intimate", "breast", "nipple", "clitoral", "dominatrix", "kissing", "foreplay", "explicit"],
        ["Explicit Sexual Acts", "Sexuality, Attraction & Intimacy"],
    ),
    "H2_Commitment": (
        ["trust", "commitment", "marriage", "ceremony", "wedding", "reassurance", "love confession"],
        ["Reconciliation, Commitments & HEA", "Bonding, Everyday Intimacy & Growth"],
    ),
    "H3_LuxuryWork": (
        ["luxury", "work", "business", "corporate", "presentation", "meeting", "deal"],
        ["Work, Wealth, Status & Institutions", "Hero's Elite Work & Business World"],
    ),
    "H4_Emotional": (
        ["emotional", "feelings", "conversation", "delusion", "confession", "reassurance"],
        ["Emotions, Cognition & Inner Life", "Bonding, Everyday Intimacy & Growth"],
    ),
    "H5_Conflict": (
        ["argument", "conflict", "hatred", "anger", "rejection", "struggles"],
        ["Conflict, Distance & Breakup Threats", "Negative Emotions & Distress"],
    ),
    "S_Procedure": (
        ["door", "phone", "knocking", "exiting", "hallway", "stairs", "desk", "table"],
        ["Domestic Spaces & Routines", "Spaces, Time, Activities & Objects"],
    ),
}

def classify_topic_to_hypothesis(row):
    """Return '; '-joined string of matched hypotheses, else 'Unclassified'."""
    label_lower = str(row.get("label", "")).lower()
    keywords_lower = str(row.get("keywords", "")).lower() if pd.notna(row.get("keywords")) else ""
    taxonomy_main = str(row.get("taxonomy_main_name", "")).lower() if pd.notna(row.get("taxonomy_main_name")) else ""

    matches = []
    for hyp, (keywords, taxonomies) in hypothesis_definitions.items():
        # Check keywords
        kw_match = any(kw.lower() in label_lower or kw.lower() in keywords_lower for kw in keywords)
        # Check taxonomies (loose string match)
        tax_match = any(tax.lower() in taxonomy_main for tax in taxonomies)
        if kw_match or tax_match:
            matches.append(hyp)
    if not matches:
        return "Unclassified"
    return "; ".join(matches)

# Classify all filtered topics
filtered_topics["hypothesis_mapping"] = filtered_topics.apply(classify_topic_to_hypothesis, axis=1)

# Print mapping summary
print("\n" + "=" * 80)
print("Topic-to-Hypothesis Mapping Summary:")
print("=" * 80)

from collections import Counter

mapping_list = filtered_topics["hypothesis_mapping"]
hypothesis_counter = Counter()
for mapping in mapping_list:
    if pd.notna(mapping) and mapping != "Unclassified":
        for hyp in mapping.split("; "):
            hypothesis_counter[hyp] += 1

for hyp in sorted(hypothesis_counter):
    print(f"  {hyp}: {hypothesis_counter[hyp]} topics")
print(f"\n  Unclassified: {(mapping_list == 'Unclassified').sum()} topics")

# Produce shortlists for every defined hypothesis label (fully filtered set)
print("\n" + "=" * 80)
print("Hypothesis Shortlists (All Tier 2 topics per hypothesis):")
print("=" * 80)
shortlist_hyps = list(hypothesis_definitions.keys())

for hyp in shortlist_hyps:
    hyp_topics = filtered_topics[filtered_topics["hypothesis_mapping"].str.contains(hyp, na=False)].copy()
    if len(hyp_topics) > 0:
        hyp_topics = hyp_topics.sort_values("abs_cliffs_top_trash", ascending=False)
        print(f"\n{hyp} ({len(hyp_topics)} topics):")
        print(hyp_topics[["label", "cliffs_top_trash", "top_median", "trash_median", "prevalence"]].to_string(index=False))

# Save both Parquet and CSV mapping tables
mapping_path = TABLE_DIR / "topic_hypothesis_mapping.parquet"
mapping_csv_path = TABLE_DIR / "topic_hypothesis_mapping.csv"
filtered_topics[["label", "topic_id", "hypothesis_mapping", "cliffs_top_trash", "top_median", "trash_median", "prevalence", "mass"]].to_parquet(mapping_path, index=False)
filtered_topics[["label", "topic_id", "hypothesis_mapping", "cliffs_top_trash", "top_median", "trash_median", "prevalence", "mass"]].to_csv(mapping_csv_path, index=False)
print(f"\n✓ Saved topic-to-hypothesis mapping to: {mapping_csv_path}")

# Scene scaffolding: extract and save procedure/structural topics
procedure_topics = filtered_topics[filtered_topics["hypothesis_mapping"].str.contains("S_Procedure", na=False)].copy()
print("\n" + "=" * 80)
print("Procedure Topics (Scene Scaffolding):")
print("=" * 80)
print(f"Found {len(procedure_topics)} procedure topics")
print("\nThese are scene anchors for qualitative sampling, not thematic content:")
print(procedure_topics[["label", "cliffs_top_trash", "top_median", "trash_median"]].to_string(index=False))

procedure_path = TABLE_DIR / "procedure_topics.parquet"
procedure_csv_path = TABLE_DIR / "procedure_topics.csv"
procedure_topics.to_parquet(procedure_path, index=False)
procedure_topics.to_csv(procedure_csv_path, index=False)
print(f"\n✓ Saved procedure topics to: {procedure_csv_path}")


Topic-to-Hypothesis Mapping & Procedure Topics Classification


RuntimeError: No Tier 2 topics or topic_leaderboard_all available for topic-to-hypothesis mapping.

## Author Dominance: Methodological Control

**Key Achievement:** We now have an objective way to identify and control for author-style topics.

### Findings:
- **41 topics** are high author-dominant (>50% from single author)
- **17 topics** are medium author-dominant (30-50%)
- Several topics are **100% from one author** (e.g., Ana Huang, Stella Rhys)

### Methodological Implications:

1. **Objective Control:** We can now say: *"We controlled for author-style topics and filtered them out."*

2. **Modeling Setup:** This sets up future modeling steps:
   - Include author fixed effects or random effects
   - Show results are not just "Ana Huang vs everyone else"
   - Demonstrate findings are about plot/content, not author style

3. **Credibility:** For romance corpora, this is the difference between:
   - "Cool plot finding" (uncontrolled)
   - **"Credible finding"** (author-controlled)

### Usage in Analysis:
- Topics flagged as `is_author_driven=True` should be:
  - Excluded from tier interpretation (they reflect author style, not tier preferences)
  - Treated as covariates in modeling
  - Documented separately as "author signature topics"

### Next Steps:
- In subgroup analysis: filter author-dominant topics before aggregation
- In modeling: include author as fixed/random effect
- In interpretation: distinguish "tier effects" from "author effects"


## Bottom-Layer Conclusions (Safe + Useful)

These are statements that are consistent with the outputs and won't collapse when moving up a level:

### 1. Specific Explicitness Topics → Lower Ratings
- **Finding:** Specific explicitness topics (not "sex in general") are associated with lower-rated books
- **Evidence:** Topics like "Intimate Breast And Nipple Kissing", "Dominatrix Session" show Trash > Top
- **Supports:** Direction of H1 (explicit content → lower ratings)
- **Note:** This is about *specific* topics, not a blanket "sex is bad" claim

### 2. Commitment/Trust Topics → Higher Ratings
- **Finding:** Top books show more commitment/trust/relationship-definition cues at topic level
- **Evidence:** Topics like "Marriage Ceremony Planning", "Trust Assurance Conversation" show Top > Trash
- **Supports:** H2 (commitment/relationship-definition → higher ratings)
- **Note:** These are topic-level patterns, not category-level aggregations

### 3. Scene Scaffolding Differences
- **Finding:** Noticeable portion of Trash–Top separation comes from scene scaffolding topics
- **Evidence:** Topics like "Knocking On Door", "Exiting Through Doorways", "Phone Ringing" show tier differences
- **Implication:** Suggests stylistic/pacing differences, not just thematic content
- **Treatment:** These are "procedure topics" for qualitative sampling, not thematic interpretation

### 4. Author Dominance is Real and Measurable
- **Finding:** 41 topics are high author-dominant (>50% from single author)
- **Implication:** Filtering author-owned topics is necessary for tier interpretation
- **Methodological Win:** Objective way to say "We controlled for author-style topics"
- **Next Step:** Include author fixed effects in modeling

### 5. Missing Probability Mass Must Be Handled Consistently
- **Finding:** Missing mass is substantial (~32% at book level, varies at segment level)
- **Implication:** Must handle consistently (book + segment) to make indices meaningful
- **Solution:** OTHER bucket approach ensures probability sums = 1.0
- **Critical:** Segment-level OTHER must be added before H6 time-course analysis

### What These Conclusions Enable:
- **Reproducible filtering:** Three-gate rule (Effect, Impact, Stability)
- **Hypothesis mapping:** Topic-to-hypothesis shortlists for subgroup analysis
- **Author control:** Objective filtering of author-style topics
- **Consistent indices:** Proper probability mass handling at all levels


In [18]:
# 12. Topic-Level Leaderboards (1.3.4)
# Create sorted tables for different perspectives

print("=" * 80)
print("Creating topic leaderboards:")
print("=" * 80)

# Merge with topic health for prevalence/mass
topic_leaderboard_all = topic_stats.merge(
    topic_health[["label", "prevalence", "mass", "concentration_ratio"]],
    on="label",
    how="left"
)

# Merge with topic_lookup to add ALL available topic metadata
# This includes: keywords, scene_summary, categories, taxonomy, Radway phases, etc.

# Core fields (always try to merge)
lookup_cols = ["topic_id", "scene_summary"]

# Keywords and labels
if "keywords" in topic_lookup.columns:
    lookup_cols.append("keywords")
else:
    print("  ⚠️  Note: 'keywords' column not found in topic_lookup - will not be included")
if "label_is_noise" in topic_lookup.columns:
    lookup_cols.append("label_is_noise")
if "label_rationale" in topic_lookup.columns:
    lookup_cols.append("label_rationale")

# Category tags (primary and secondary) - useful for understanding topic themes
if "primary_categories" in topic_lookup.columns:
    lookup_cols.append("primary_categories")
if "secondary_categories" in topic_lookup.columns:
    lookup_cols.append("secondary_categories")

# Taxonomy fields (main category) - hierarchical classification
if "taxonomy_main_name" in topic_lookup.columns:
    lookup_cols.append("taxonomy_main_name")
if "taxonomy_main_group" in topic_lookup.columns:
    lookup_cols.append("taxonomy_main_group")
if "taxonomy_confidence" in topic_lookup.columns:
    lookup_cols.append("taxonomy_confidence")
if "taxonomy_is_noise" in topic_lookup.columns:
    lookup_cols.append("taxonomy_is_noise")

# Taxonomy fields (secondary category - may be sparse)
if "taxonomy_secondary_name" in topic_lookup.columns:
    lookup_cols.append("taxonomy_secondary_name")
if "taxonomy_secondary_group" in topic_lookup.columns:
    lookup_cols.append("taxonomy_secondary_group")

# Radway phase fields - narrative structure classification
if "radway_main_name" in topic_lookup.columns:
    lookup_cols.append("radway_main_name")
if "radway_phase_name" in topic_lookup.columns:
    lookup_cols.append("radway_phase_name")
if "radway_phase" in topic_lookup.columns:
    lookup_cols.append("radway_phase")
if "radway_confidence" in topic_lookup.columns:
    lookup_cols.append("radway_confidence")
if "radway_is_none" in topic_lookup.columns:
    lookup_cols.append("radway_is_none")

# Get topic_id from topic_stats to merge
merge_cols = [c for c in lookup_cols if c != "label"]  # Remove label since we're merging on topic_id
topic_leaderboard_all = topic_leaderboard_all.merge(
    topic_lookup[merge_cols].drop_duplicates("topic_id"),
    on="topic_id",
    how="left"
)

# Report what was merged (grouped by category for clarity)
merged_cols = [c for c in merge_cols if c != "topic_id"]
core_fields = ["scene_summary", "keywords"] if "keywords" in merged_cols else ["scene_summary"]
category_fields = [c for c in merged_cols if "categor" in c.lower()]
taxonomy_fields = [c for c in merged_cols if "taxonomy" in c.lower()]
radway_fields = [c for c in merged_cols if "radway" in c.lower()]
other_fields = [c for c in merged_cols if c not in core_fields + category_fields + taxonomy_fields + radway_fields]

print(f"  ✓ Merged from topic_lookup:")
if core_fields:
    print(f"    Core: {', '.join(core_fields)}")
if category_fields:
    print(f"    Categories: {', '.join(category_fields)}")
if taxonomy_fields:
    print(f"    Taxonomy: {', '.join(taxonomy_fields)}")
if radway_fields:
    print(f"    Radway: {', '.join(radway_fields)}")
if other_fields:
    print(f"    Other: {', '.join(other_fields)}")

# 1. Most Top-associated topics (highest median/mean in Top)
# Include keywords and rationale for noise checking
display_cols = ["label", "topic_id", "top_median", "top_mean", "trash_median", "cliffs_top_trash", "prevalence", "mass"]
if "keywords" in topic_leaderboard_all.columns:
    display_cols.append("keywords")
if "label_rationale" in topic_leaderboard_all.columns:
    display_cols.append("label_rationale")
if "label_is_noise" in topic_leaderboard_all.columns:
    display_cols.append("label_is_noise")

top_associated = topic_leaderboard_all.nlargest(50, "top_median")[display_cols].copy()
top_associated = top_associated.sort_values("top_median", ascending=False)

# 2. Most Trash-associated topics (highest median/mean in Trash)
trash_display_cols = ["label", "topic_id", "trash_median", "trash_mean", "top_median", "cliffs_top_trash", "prevalence", "mass"]
if "keywords" in topic_leaderboard_all.columns:
    trash_display_cols.append("keywords")
if "label_rationale" in topic_leaderboard_all.columns:
    trash_display_cols.append("label_rationale")
if "label_is_noise" in topic_leaderboard_all.columns:
    trash_display_cols.append("label_is_noise")

trash_associated = topic_leaderboard_all.nlargest(50, "trash_median")[trash_display_cols].copy()
trash_associated = trash_associated.sort_values("trash_median", ascending=False)

# 3. Most Medium-peaked topics (highest in Middle relative to others)
topic_leaderboard_all["middle_relative"] = (
    topic_leaderboard_all["middle_median"] / 
    (topic_leaderboard_all[["top_median", "trash_median"]].max(axis=1) + 1e-10)
)
middle_peaked = topic_leaderboard_all.nlargest(50, "middle_relative")[
    ["label", "topic_id", "middle_median", "top_median", "trash_median", "middle_relative", "prevalence", "mass"]
].copy()
middle_peaked = middle_peaked.sort_values("middle_relative", ascending=False)

# 4. Topics with biggest Top–Trash separation (largest effect size)
# Use absolute value of Cliff's delta
if "abs_cliffs_top_trash" not in topic_leaderboard_all.columns:
    topic_leaderboard_all["abs_cliffs_top_trash"] = topic_leaderboard_all["cliffs_top_trash"].abs()

effect_display_cols = ["label", "topic_id", "cliffs_top_trash", "top_median", "trash_median", "mw_top_trash_p", "prevalence", "mass"]
if "keywords" in topic_leaderboard_all.columns:
    effect_display_cols.append("keywords")
if "label_rationale" in topic_leaderboard_all.columns:
    effect_display_cols.append("label_rationale")
if "label_is_noise" in topic_leaderboard_all.columns:
    effect_display_cols.append("label_is_noise")
if "abs_cliffs_top_trash" not in effect_display_cols:
    effect_display_cols.append("abs_cliffs_top_trash")

# Select cols including abs_cliffs_top_trash for sorting and display, so sort doesn't error
effect_sizes = topic_leaderboard_all.nlargest(50, "abs_cliffs_top_trash")[effect_display_cols].copy()
effect_sizes = effect_sizes.sort_values("abs_cliffs_top_trash", ascending=False)

# Save all leaderboards
print("\nSaving leaderboards...")
topic_leaderboard_all_path = TABLE_DIR / "topic_leaderboard_all.parquet"
topic_leaderboard_all.to_parquet(topic_leaderboard_all_path, index=False)
topic_leaderboard_all_csv_path = TABLE_DIR / "topic_leaderboard_all.csv"
topic_leaderboard_all.to_csv(topic_leaderboard_all_csv_path, index=False)
print(f"  ✓ topic_leaderboard_all.parquet ({len(topic_leaderboard_all)} topics)")
print(f"  ✓ topic_leaderboard_all.csv ({len(topic_leaderboard_all)} topics)")

top_associated_path = TABLE_DIR / "topic_leaderboard_top_associated.parquet"
top_associated.to_parquet(top_associated_path, index=False)
top_associated_csv_path = TABLE_DIR / "topic_leaderboard_top_associated.csv"
top_associated.to_csv(top_associated_csv_path, index=False)
print(f"  ✓ topic_leaderboard_top_associated.parquet (top 50)")
print(f"  ✓ topic_leaderboard_top_associated.csv (top 50)")

trash_associated_path = TABLE_DIR / "topic_leaderboard_trash_associated.parquet"
trash_associated.to_parquet(trash_associated_path, index=False)
trash_associated_csv_path = TABLE_DIR / "topic_leaderboard_trash_associated.csv"
trash_associated.to_csv(trash_associated_csv_path, index=False)
print(f"  ✓ topic_leaderboard_trash_associated.parquet (top 50)")
print(f"  ✓ topic_leaderboard_trash_associated.csv (top 50)")

effect_sizes_path = TABLE_DIR / "topic_leaderboard_effect_sizes.parquet"
effect_sizes.to_parquet(effect_sizes_path, index=False)
effect_sizes_csv_path = TABLE_DIR / "topic_leaderboard_effect_sizes.csv"
effect_sizes.to_csv(effect_sizes_csv_path, index=False)
print(f"  ✓ topic_leaderboard_effect_sizes.parquet (top 50)")
print(f"  ✓ topic_leaderboard_effect_sizes.csv (top 50)")

# Display summaries
print("\n" + "=" * 80)
print("Top 10 Most Top-Associated Topics:")
print("=" * 80)
print(top_associated[["label", "top_median", "trash_median", "cliffs_top_trash"]].head(10).to_string(index=False))

print("\n" + "=" * 80)
print("Top 10 Most Trash-Associated Topics:")
print("=" * 80)
print(trash_associated[["label", "trash_median", "top_median", "cliffs_top_trash"]].head(10).to_string(index=False))

print("\n" + "=" * 80)
print("Top 10 Topics by Effect Size (Top vs Trash):")
print("=" * 80)
# Before displaying, check if abs_cliffs_top_trash in effect_sizes columns
if "abs_cliffs_top_trash" not in effect_sizes.columns:
    print("WARNING: abs_cliffs_top_trash missing from effect_sizes DataFrame. Effect size sorting might not work as intended.")
print(effect_sizes[["label", "cliffs_top_trash", "top_median", "trash_median"]].head(10).to_string(index=False))


Creating topic leaderboards:
  ✓ Merged from topic_lookup:
    Core: scene_summary, keywords
    Categories: primary_categories, secondary_categories
    Taxonomy: taxonomy_main_name, taxonomy_main_group, taxonomy_confidence, taxonomy_is_noise, taxonomy_secondary_name, taxonomy_secondary_group
    Radway: radway_main_name, radway_phase_name, radway_phase, radway_confidence, radway_is_none
    Other: label_is_noise

Saving leaderboards...
  ✓ topic_leaderboard_all.parquet (343 topics)
  ✓ topic_leaderboard_all.csv (343 topics)
  ✓ topic_leaderboard_top_associated.parquet (top 50)
  ✓ topic_leaderboard_top_associated.csv (top 50)
  ✓ topic_leaderboard_trash_associated.parquet (top 50)
  ✓ topic_leaderboard_trash_associated.csv (top 50)
  ✓ topic_leaderboard_effect_sizes.parquet (top 50)
  ✓ topic_leaderboard_effect_sizes.csv (top 50)

Top 10 Most Top-Associated Topics:
                              label  top_median  trash_median  cliffs_top_trash
         OTHER (Missing/Noise Mass)    0

In [19]:
# 13. Two-Tier Topic Filtering with Two-Gate Rule
# 
# RATIONALE: With only 30 books per tier, FDR-corrected p-values are underpowered.
# We use a two-gate rule for meaningful interpretation:
#   - Gate 1 (Effect): |Cliff's δ| ≥ 0.20 (meaningful effect size)
#   - Gate 2 (Impact): mass ≥ 0.002 OR |Top–Trash mean diff| ≥ 0.001 (meaningful impact)
#
# Two-tier structure:
#   - Tier 1 (High Confidence): Large effects (|δ| ≥ 0.35) + raw p < 0.05 + both gates
#   - Tier 2 (Exploratory): Moderate effects (|δ| ≥ 0.20) + both gates (no p-value filter)
#
# This is appropriate for hypothesis-generating exploratory research.

print("=" * 80)
print("Two-Tier Topic Filtering with Two-Gate Rule")
print("=" * 80)

# Note on sample size limitation and two-gate rule
print("""
NOTE: Statistical Power Limitation & Two-Gate Rule
--------------------------------------------------
With n=30 books per tier, even large effect sizes (|δ| > 0.35) cannot 
survive FDR correction. The smallest adjusted p-value is ~0.20.

Solution: Two-gate rule for meaningful interpretation:
  - Gate 1 (Effect): |Cliff's δ| ≥ 0.20 (meaningful effect size)
  - Gate 2 (Impact): mass ≥ 0.002 OR |Top–Trash mean diff| ≥ 0.001 (meaningful impact)

Two-tier structure:
  - Tier 1: High-confidence topics (large effects + raw significance + both gates)
  - Tier 2: Exploratory topics (moderate effects + both gates, no p-value filter)
""")

# Compute mean difference for impact gate
if "mean_diff_top_trash" not in topic_leaderboard_all.columns:
    topic_leaderboard_all["mean_diff_top_trash"] = (
        topic_leaderboard_all["top_mean"] - topic_leaderboard_all["trash_mean"]
    ).abs()

# Define gate thresholds
EFFECT_THRESHOLD = 0.20  # Gate 1: meaningful effect size
MASS_THRESHOLD = 0.002  # Gate 2a: meaningful mass
MEAN_DIFF_THRESHOLD = 0.001  # Gate 2b: meaningful mean difference

# Apply two-gate rule: both gates must pass
# Gate 1: |Cliff's δ| ≥ EFFECT_THRESHOLD
# Gate 2: mass ≥ MASS_THRESHOLD OR |Top–Trash mean diff| ≥ MEAN_DIFF_THRESHOLD
gate1_mask = topic_leaderboard_all["abs_cliffs_top_trash"] >= EFFECT_THRESHOLD
gate2_mask = (
    (topic_leaderboard_all["mass"] >= MASS_THRESHOLD) |
    (topic_leaderboard_all["mean_diff_top_trash"] >= MEAN_DIFF_THRESHOLD)
)
two_gate_mask = gate1_mask & gate2_mask & topic_leaderboard_all["label"].notna()

print(f"\nTwo-Gate Rule Summary:")
print(f"  Gate 1 (Effect |δ| ≥ {EFFECT_THRESHOLD}): {gate1_mask.sum()} topics")
print(f"  Gate 2 (Impact: mass ≥ {MASS_THRESHOLD} OR |mean diff| ≥ {MEAN_DIFF_THRESHOLD}): {gate2_mask.sum()} topics")
print(f"  Both gates passed: {two_gate_mask.sum()} topics")

# ============================================================================
# TIER 1: High Confidence Topics
# Criteria: |δ| ≥ 0.35 AND raw p < 0.05 AND both gates passed
# ============================================================================
print("\n" + "=" * 80)
print("TIER 1: High Confidence Topics")
print("=" * 80)
print(f"Criteria: |Cliff's δ| ≥ 0.35, raw p < 0.05, both gates passed")
print(f"  (Effect: |δ| ≥ {EFFECT_THRESHOLD}, Impact: mass ≥ {MASS_THRESHOLD} OR |mean diff| ≥ {MEAN_DIFF_THRESHOLD})")

tier1 = topic_leaderboard_all[
    two_gate_mask &
    (topic_leaderboard_all["abs_cliffs_top_trash"] >= 0.35) &
    (topic_leaderboard_all["mw_top_trash_p"] < 0.05)
].copy().sort_values("abs_cliffs_top_trash", ascending=False)

# Add direction indicator
tier1["direction"] = tier1["cliffs_top_trash"].apply(
    lambda x: "Top ↑" if x > 0 else "Trash ↑"
)

print(f"\nTier 1 topics found: {len(tier1)}")

# Save Tier 1
tier1_path = TABLE_DIR / "topic_leaderboard_tier1_high_confidence.parquet"
tier1.to_parquet(tier1_path, index=False)
tier1_csv_path = TABLE_DIR / "topic_leaderboard_tier1_high_confidence.csv"
tier1.to_csv(tier1_csv_path, index=False)
print(f"✓ Saved to: {tier1_csv_path}")

# Display Tier 1
if len(tier1) > 0:
    display_cols_t1 = ["label", "direction", "cliffs_top_trash", "mw_top_trash_p", 
                       "mass", "mean_diff_top_trash", "top_median", "trash_median"]
    # Include all available metadata fields
    metadata_fields = ["keywords", "scene_summary", "primary_categories", "secondary_categories",
                     "taxonomy_main_name", "taxonomy_main_group", "taxonomy_confidence",
                     "radway_main_name", "radway_phase_name", "radway_phase", "radway_confidence",
                     "label_is_noise", "taxonomy_is_noise", "radway_is_none"]
    for field in metadata_fields:
        if field in tier1.columns and field not in display_cols_t1:
            display_cols_t1.append(field)
    print("\n" + tier1[display_cols_t1].to_string(index=False))
else:
    print("(No topics in Tier 1)")

# ============================================================================
# TIER 2: Exploratory Topics  
# Criteria: |δ| ≥ 0.20 AND both gates passed (no p-value filter)
# ============================================================================
print("\n" + "=" * 80)
print("TIER 2: Exploratory Topics")
print("=" * 80)
print(f"Criteria: |Cliff's δ| ≥ {EFFECT_THRESHOLD}, both gates passed (no p-value filter)")
print(f"  (Effect: |δ| ≥ {EFFECT_THRESHOLD}, Impact: mass ≥ {MASS_THRESHOLD} OR |mean diff| ≥ {MEAN_DIFF_THRESHOLD})")

tier2 = topic_leaderboard_all[
    two_gate_mask &
    (topic_leaderboard_all["abs_cliffs_top_trash"] >= EFFECT_THRESHOLD)
].copy().sort_values("abs_cliffs_top_trash", ascending=False)

# Add direction indicator
tier2["direction"] = tier2["cliffs_top_trash"].apply(
    lambda x: "Top ↑" if x > 0 else "Trash ↑"
)

# Mark which ones are also in Tier 1
tier2["tier"] = tier2["label"].apply(
    lambda x: "Tier 1" if x in tier1["label"].values else "Tier 2"
)

print(f"\nTier 2 topics found: {len(tier2)} (includes {len(tier1)} from Tier 1)")

# Save Tier 2
tier2_path = TABLE_DIR / "topic_leaderboard_tier2_exploratory.parquet"
tier2.to_parquet(tier2_path, index=False)
tier2_csv_path = TABLE_DIR / "topic_leaderboard_tier2_exploratory.csv"
tier2.to_csv(tier2_csv_path, index=False)
print(f"✓ Saved to: {tier2_csv_path}")

# Display Tier 2 (top 30)
if len(tier2) > 0:
    display_cols_t2 = ["label", "tier", "direction", "cliffs_top_trash", 
                       "mass", "mean_diff_top_trash", "mw_top_trash_p", "prevalence"]
    # Include all available metadata fields
    metadata_fields = ["keywords", "scene_summary", "primary_categories", "secondary_categories",
                     "taxonomy_main_name", "taxonomy_main_group", "taxonomy_confidence",
                     "radway_main_name", "radway_phase_name", "radway_phase", "radway_confidence",
                     "label_is_noise", "taxonomy_is_noise", "radway_is_none"]
    for field in metadata_fields:
        if field in tier2.columns and field not in display_cols_t2:
            display_cols_t2.append(field)
    print(f"\nTop 30 by effect size:")
    print(tier2[display_cols_t2].head(30).to_string(index=False))
else:
    print("(No topics in Tier 2)")

# ============================================================================
# Summary Statistics
# ============================================================================
print("\n" + "=" * 80)
print("Summary")
print("=" * 80)
print(f"Tier 1 (High Confidence): {len(tier1)} topics")
print(f"  - Top-associated (δ > 0): {(tier1['cliffs_top_trash'] > 0).sum()}")
print(f"  - Trash-associated (δ < 0): {(tier1['cliffs_top_trash'] < 0).sum()}")
print(f"\nTier 2 (Exploratory): {len(tier2)} topics")
print(f"  - Top-associated (δ > 0): {(tier2['cliffs_top_trash'] > 0).sum()}")
print(f"  - Trash-associated (δ < 0): {(tier2['cliffs_top_trash'] < 0).sum()}")

# Also save the combined "filtered" for backwards compatibility
# Use Tier 2 as the main filtered output (broader exploratory set)
filtered = tier2.copy()
filtered_path = TABLE_DIR / "topic_leaderboard_filtered.parquet"
filtered.to_parquet(filtered_path, index=False)
filtered_csv_path = TABLE_DIR / "topic_leaderboard_filtered.csv"
filtered.to_csv(filtered_csv_path, index=False)
print(f"\n✓ Saved combined filtered leaderboard to: {filtered_csv_path} ({len(filtered)} topics)")

# Verify that keywords and scene_summary are included in saved files
print("\n" + "=" * 80)
print("Columns included in saved tables:")
print("=" * 80)
saved_cols = ["label", "scene_summary"]
if "keywords" in filtered.columns:
    saved_cols.append("keywords")
    print("  ✓ keywords: Included")
else:
    print("  ⚠️  keywords: Not available in topic_lookup")
if "scene_summary" in filtered.columns:
    print("  ✓ scene_summary: Included")
else:
    print("  ⚠️  scene_summary: Not available")
print(f"\nAll saved tables (tier1, tier2, filtered) include: {', '.join(saved_cols)}")
print("  + statistical metrics (cliffs_top_trash, p-values, medians, etc.)")


Two-Tier Topic Filtering with Two-Gate Rule

NOTE: Statistical Power Limitation & Two-Gate Rule
--------------------------------------------------
With n=30 books per tier, even large effect sizes (|δ| > 0.35) cannot 
survive FDR correction. The smallest adjusted p-value is ~0.20.

Solution: Two-gate rule for meaningful interpretation:
  - Gate 1 (Effect): |Cliff's δ| ≥ 0.20 (meaningful effect size)
  - Gate 2 (Impact): mass ≥ 0.002 OR |Top–Trash mean diff| ≥ 0.001 (meaningful impact)

Two-tier structure:
  - Tier 1: High-confidence topics (large effects + raw significance + both gates)
  - Tier 2: Exploratory topics (moderate effects + both gates, no p-value filter)


Two-Gate Rule Summary:
  Gate 1 (Effect |δ| ≥ 0.2): 136 topics
  Gate 2 (Impact: mass ≥ 0.002 OR |mean diff| ≥ 0.001): 114 topics
  Both gates passed: 45 topics

TIER 1: High Confidence Topics
Criteria: |Cliff's δ| ≥ 0.35, raw p < 0.05, both gates passed
  (Effect: |δ| ≥ 0.2, Impact: mass ≥ 0.002 OR |mean diff| ≥ 0.001)


In [20]:
# 14. Author as "Shadow Confounder" (1.5)
# Check whether topics are dominated by 1-2 authors

print("=" * 80)
print("Checking author dominance per topic:")
print("=" * 80)

# Load author information from goodreads
if "Author" in goodreads.columns:
    # Merge author into book_wide if not already present
    if "Author" not in book_wide.columns:
        book_wide = book_wide.merge(
            goodreads[["book_id", "Author"]],
            on="book_id",
            how="left"
        )
    
    # Merge author into topic probabilities
    book_topic_probs_with_author = book_topic_probs_all.merge(
        book_wide[["book_id", "Author"]],
        on="book_id",
        how="left"
    )
    
    print(f"Author information merged. Books with author: {book_topic_probs_with_author['Author'].notna().sum() / len(book_topic_probs_with_author) * 100:.1f}%")
    
    def compute_author_dominance(df: pd.DataFrame) -> pd.DataFrame:
        """Compute author dominance metrics for each topic."""
        results = []
        
        unique_labels = df["label"].dropna().unique()
        print(f"Computing author dominance for {len(unique_labels)} topics...")
        
        for label in unique_labels:
            topic_data = df[df["label"] == label].copy()
            
            if len(topic_data) == 0 or topic_data["Author"].isna().all():
                continue
            
            topic_id = topic_data["topic_id"].iloc[0]
            
            # Compute topic prevalence per author
            # Use books where topic prob > threshold
            threshold = 0.001
            topic_books = topic_data[topic_data["prob"] > threshold].copy()
            
            if len(topic_books) == 0:
                continue
            
            # Count books per author for this topic
            author_counts = topic_books.groupby("Author")["book_id"].nunique().sort_values(ascending=False)
            
            # Dominance metrics
            total_books = len(topic_books["book_id"].unique())
            top_author = author_counts.index[0] if len(author_counts) > 0 else None
            top_author_count = author_counts.iloc[0] if len(author_counts) > 0 else 0
            top_author_share = top_author_count / total_books if total_books > 0 else 0
            
            top2_author_count = author_counts.head(2).sum() if len(author_counts) >= 2 else top_author_count
            top2_author_share = top2_author_count / total_books if total_books > 0 else 0
            
            # Check if topic is tier-stable vs author-driven
            # Tier-stable: similar prevalence across tiers regardless of author
            # Author-driven: dominated by one author who might be in one tier
            
            # Compute prevalence by tier
            tier_prevalence = topic_books.groupby("rating_tier")["book_id"].nunique() / topic_books.groupby("rating_tier")["book_id"].nunique().sum() if len(topic_books) > 0 else {}
            
            # Flag: if top author accounts for >50% of books, flag as potentially author-driven
            is_author_driven = top_author_share > 0.5
            
            # Check if top author's books are concentrated in one tier
            if top_author and is_author_driven:
                top_author_books = topic_books[topic_books["Author"] == top_author]
                if len(top_author_books) > 0:
                    top_author_tier_dist = top_author_books["rating_tier"].value_counts(normalize=True)
                    top_author_tier_concentration = top_author_tier_dist.max()
                else:
                    top_author_tier_concentration = 0
            else:
                top_author_tier_concentration = 0
            
            result = {
                "topic_id": topic_id,
                "label": label,
                "total_books_with_topic": total_books,
                "n_authors": len(author_counts),
                "top_author": top_author,
                "top_author_count": top_author_count,
                "top_author_share": top_author_share,
                "top2_author_share": top2_author_share,
                "is_author_driven": is_author_driven,
                "top_author_tier_concentration": top_author_tier_concentration,
                "author_dominance_flag": "high" if top_author_share > 0.5 else "medium" if top_author_share > 0.3 else "low"
            }
            
            results.append(result)
        
        return pd.DataFrame(results)
    
    # Compute author dominance
    author_dominance = compute_author_dominance(book_topic_probs_with_author)
    
    # Merge with topic stats
    author_dominance_full = author_dominance.merge(
        topic_leaderboard_all[["label", "cliffs_top_trash", "top_median", "trash_median", "prevalence"]],
        on="label",
        how="left"
    )
    
    # Save
    author_dominance_path = TABLE_DIR / "topic_author_dominance.parquet"
    author_dominance_full.to_parquet(author_dominance_path, index=False)
    author_dominance_csv_path = TABLE_DIR / "topic_author_dominance.csv"
    author_dominance_full.to_csv(author_dominance_csv_path, index=False)
    print(f"\n✓ Saved author dominance analysis to: {author_dominance_path}")
    print(f"✓ Saved author dominance analysis (CSV) to: {author_dominance_csv_path}")
    
    # Summary statistics
    print("\n" + "=" * 80)
    print("Author Dominance Summary:")
    print("=" * 80)
    print(f"Topics analyzed: {len(author_dominance)}")
    print(f"Topics with high author dominance (>50%): {author_dominance['is_author_driven'].sum()}")
    print(f"Topics with medium author dominance (30-50%): {((author_dominance['top_author_share'] > 0.3) & (author_dominance['top_author_share'] <= 0.5)).sum()}")
    
    print("\nTop 10 topics by author dominance:")
    print(author_dominance.nlargest(10, "top_author_share")[
        ["label", "top_author", "top_author_share", "top2_author_share", "author_dominance_flag"]
    ].to_string(index=False))
    
    # Flag topics that are both statistically significant AND author-driven
    if len(filtered) > 0:
        filtered_with_author = filtered.merge(
            author_dominance[["label", "is_author_driven", "top_author", "top_author_share"]],
            on="label",
            how="left"
        )
        
        print("\n" + "=" * 80)
        print("Filtered Topics with Author Dominance Flags:")
        print("=" * 80)
        author_driven_filtered = filtered_with_author[filtered_with_author["is_author_driven"] == True]
        print(f"Topics that are both significant AND author-driven: {len(author_driven_filtered)}")
        if len(author_driven_filtered) > 0:
            print("\nThese topics may reflect author style rather than tier preferences:")
            print(author_driven_filtered[["label", "top_author", "top_author_share", "cliffs_top_trash"]].to_string(index=False))
    
else:
    print("⚠️  'Author' column not found in goodreads data. Skipping author dominance analysis.")

Checking author dominance per topic:
Author information merged. Books with author: 100.0%
Computing author dominance for 343 topics...

✓ Saved author dominance analysis to: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/correlation_analysis/01_topic_analysis/tables/topic_author_dominance.parquet
✓ Saved author dominance analysis (CSV) to: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/correlation_analysis/01_topic_analysis/tables/topic_author_dominance.csv

Author Dominance Summary:
Topics analyzed: 324
Topics with high author dominance (>50%): 41
Topics with medium author dominance (30-50%): 17

Top 10 topics by author dominance:
                                  label     top_author  top_author_share  top2_author_share author_dominance_flag
          Unclear Feelings Conversation      ana huang               1.0                1.0                  high
          Doorway Questi